In [ ]:
import torch
print("CUDA Available: ", torch.cuda.is_available())
print("CUDA Device Name: ", torch.cuda.get_device_name(0))
torch.cuda.empty_cache()

# Verify CUDA
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using Device: {device}")

# Load TSpec Data

In [ ]:
import os
import re
from bs4 import BeautifulSoup
from markdown import markdown
from app.utils.settings import DATA_DIRECTORY, CHUNKS_FILE
from app.utils.chunking import save_chunks

# This function loads markdown files, preserves formatting, and converts HTML tables to readable markdown/plaintext.
def load_tspec_data(directory):
    """
    Load .md files while preserving formatting (text and tables) in a readable form.
    For each .md file (filtered to Rel-18 / 28_series like the previous loader),
    returns a dict with:
      - release, series, spec (filename without .md)
      - original_md: original markdown text
      - processed_text: extracted text with tables converted to markdown/plaintext
      - html: HTML generated from the markdown (useful for debug/visualization)
    """
    data = []

    for release in os.listdir(directory):
        release_path = os.path.join(directory, release)
        if os.path.isdir(release_path) and release == "Rel-18":
            for series in os.listdir(release_path):
                series_path = os.path.join(release_path, series)
                if os.path.isdir(series_path) and series == "28_series":
                    for file in os.listdir(series_path):
                        if file.endswith('.md'):
                            file_path = os.path.join(series_path, file)
                            with open(file_path, 'r', encoding='utf-8') as f:
                                content = f.read()
                                spec_name = file[:-3]
                                
                                # Apply the same filter as the previous loader
                                if release == "Rel-18" and series == "28_series" and spec_name[:5] == "28532":

                                    # Convert markdown to HTML (support tables and fenced code)
                                    html = markdown(content, extensions=["tables", "fenced_code", "codehilite"])

                                    # Use BeautifulSoup to find HTML tables and convert them to readable markdown/plaintext
                                    soup = BeautifulSoup(html, "html.parser")

                                    # Extract text from the resulting HTML, preserving line breaks
                                    processed_text = soup.get_text(separator="\n", strip=True)

                                    # Normalize multiple blank lines
                                    processed_text = re.sub(r"\n\s*\n+", "\n\n", processed_text).strip()

                                    data.append({
                                        "release": release,
                                        "series": series,
                                        "spec": spec_name,
                                        "original_md": content,
                                        "processed_text": processed_text,
                                        "html": str(soup)
                                    })
    return data

In [ ]:
# directory_path = '../../../../../Dataset/TSpec-LLM/3GPP-clean'
directory_path = DATA_DIRECTORY
tspec_data = load_tspec_data(directory_path)

In [ ]:
print(f"Sample document:\n{tspec_data[0]['original_md'][:10000]}")

In [ ]:
# print(f"Total documents loaded: {len(tspec_data)}")
# print(f"Sample document: {tspec_data[0]['processed_text']}")
# print(f"Sample document: {tspec_data[0]['html']}")


# Build chunks (isolate Tables)

## Splitter configuration

In [ ]:
from langchain.text_splitter import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

In [ ]:
headers_to_split_on = [
    (r"\n[0-9]+\.?[0-9]*\s", "Section"),  # Matches "1 ", "4.1 ", "6.3.2 " etc.
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3")
]

markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on,
    strip_headers=False   # Keep headers inside the chunks
)

chunk_size = 500  
chunk_overlap = 100

separators = [
    r"\n[0-9]+\.?[0-9]*\s",   # High priority: section numbers
    "\n\n",
    "\n",
    " ",
    ""
]

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    separators=separators
)

## Function to detect and extract tables

In [ ]:
from typing import List, Dict, Tuple

In [ ]:
def isolate_tables(content: str) -> Tuple[str, List[str]]:
    """
    Detects and extracts all tables from the processed content.
    Returns: (content without tables, list of extracted tables as raw strings)

    Tables in processed Markdown text (after BeautifulSoup .get_text()) usually appear as:
    +----------------------+--------------------+
    | Header1             | Header2           |
    +----------------------+--------------------+
    | Row1                | Row2              |
    etc.

    We use regex to capture blocks that:
    - Start and end with border lines: +[-=]+
    - Contain lines with | separators
    - Are continuous blocks
    """
    # Regex to match Markdown table-like blocks in plain text
    table_pattern = r"(^\+[-+=]+\+$(\n^\|[^|]*\|.*\n?)+^\+[-+=]+\+$)"

    # MULTILINE + DOTALL to handle multi-line blocks
    matches = re.finditer(table_pattern, content, re.MULTILINE | re.DOTALL)

    tables = []
    content_without_tables = content
    offset = 0  # Adjust positions when removing

    for match in matches:
        table_text = match.group(0).strip()
        if len(table_text) > 100:  # Ignore very small "tables" (noise)
            tables.append(table_text)

        # Remove the table from original content (adjust offset)
        start, end = match.start() - offset, match.end() - offset
        content_without_tables = content_without_tables[:start] + content_without_tables[end:]
        offset += (end - start)

    # Clean up multiple newlines after removal
    content_without_tables = re.sub(r"\n\s*\n+", "\n\n", content_without_tables).strip()

    return content_without_tables, tables

In [ ]:
table_pattern = r"(^\+[-+=]+\+$(\n^\|[^|]*\|.*\n?)+^\+[-+=]+\+$)"
print(table_pattern)

## Divide text and tables into chunks

In [ ]:
def divide_into_chunks(tspec_data: List[Dict]) -> List[Dict]:
    """
    Splits the documents into chunks, separating tables as individual chunks.
    
    Returns a list of chunk dictionaries ready for embedding and vector store.
    """
    dataset_chunks = []

    for document in tspec_data:
        release = document['release']
        series = document['series']
        spec = document['spec']
        content = document['original_md']

        # Extract ALL tables
        content_without_tables, tables = isolate_tables(content)

        # Split main content (without tables) by headers
        header_chunks = markdown_splitter.split_text(content_without_tables)

        for header_chunk in header_chunks:
            # Use .page_content (important!)
            char_chunks = text_splitter.split_text(header_chunk.page_content)

            for chunk_text in char_chunks:
                if len(chunk_text.strip()) < 60:  # Skip insignificant fragments
                    continue

                dataset_chunks.append({
                    "release": release,
                    "series": series,
                    "spec": spec,
                    "text": chunk_text.strip(),           # Content that will be embedded
                    "processed_text": None,               # Only used for tables
                    "is_table": False
                })

        # Add each table as a separate chunk
        for table in tables:
            if len(table.strip()) > 150:  # Avoid very small tables
                dataset_chunks.append({
                    "release": release,
                    "series": series,
                    "spec": spec,
                    "text": table.strip(),
                    "processed_text": table.strip(),      # Keep original table text
                    "is_table": True
                })

    return dataset_chunks

In [ ]:
tspec_chunks = divide_into_chunks(tspec_data)

In [ ]:
tables = [table for table in tspec_chunks if table['is_table']]

In [ ]:
print(f"Total chunks created: {len(tables)}")
print(f"Example chunk:\n{tables[0]['is_table']}")
print(f"Example chunk:\n{tables[0]['text']}")

# Build chunks (without isolate tables)

## Divide into chunks

In [ ]:
from langchain.text_splitter import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
import re

# Configure headers for splitting
headers_to_split_on = [
    (r"\n[0-9]+\.?[0-9]*\s", "Section"),  # Captura "1 ", "4.1 ", etc.
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3")
]

# Initialize the MarkdownHeaderTextSplitter
markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on, strip_headers=False)

# Configure the RecursiveCharacterTextSplitter
chunk_size = 1000  # Ajustado para ~500 palavras
chunk_overlap = 100
separators = [r"\n[0-9]+\.?[0-9]*\s", "\n\n", "\n", " ", ""]
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    separators=separators
)

# Function to divide content into chunks
def divide_into_chunks(tspec_data):
    dataset_chunks = []

    for document in tspec_data:
        release = document['release']
        series = document['series']
        spec = document['spec']
        content = document['original_md']
        
        # Split by Markdown headers
        header_chunks = markdown_splitter.split_text(content)
        
        # Further split the chunks by characters
        for header_chunk in header_chunks:
            char_chunks = text_splitter.split_text(header_chunk.page_content)
            for chunk in char_chunks:
                dataset_chunks.append({
                    'release': release,
                    'series': series,
                    'spec': spec,
                    'content': chunk
                })

    return dataset_chunks

In [ ]:
tspec_chunks = divide_into_chunks(tspec_data)

In [ ]:
# Check the result
print(f"Total chunks created: {len(tspec_chunks)}")
print(f"Example chunk:\n {tspec_chunks[0]}")

In [ ]:
chunks_path = CHUNKS_FILE
save_chunks(tspec_chunks, chunks_path)